# 06_model_registry.ipynb

This notebook registers the SageMaker-compatible XGBoost model in the SageMaker Model Registry.

The workflow loads model, evaluation, and Clarify metadata, creates a Model Package Group if needed, registers a new model version, and stores registry metadata in Amazon S3.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and AWS SDK clients.

This notebook uses `boto3` directly instead of the SageMaker Python SDK.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    SAGEMAKER_MODEL_METADATA_KEY,
    EVALUATION_REPORT_KEY,
    ECALUATION_REPORT_S3_URI,
    MODEL_PACKAGE_GROUP_NAME,
    MODEL_PACKAGE_GROUP_DESCRIPTION,
    MODEL_APPROVAL_STATUS,
    MODEL_REGISTRY_METADATA_KEY,
    CLARIFY_METADATA_PREFIX,
)

## Initialize AWS Clients

Initialize the AWS clients required for Amazon S3 and SageMaker.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)

## Define Registry Configuration

Define local paths and timestamps used for storing registry metadata.

The Model Package Group uses a stable name, while each registered model package becomes a new version inside the group.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

LOCAL_REGISTRY_DIR = Path("data/registry")
LOCAL_REGISTRY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_REGISTRY_METADATA_PATH = LOCAL_REGISTRY_DIR / "model_registry_metadata.json"

print(f"Model package group: {MODEL_PACKAGE_GROUP_NAME}")
print(f"Approval status: {MODEL_APPROVAL_STATUS}")

## Load Model Metadata

Load the SageMaker Model metadata created in the model creation notebook.

The metadata contains the model artifact S3 URI, container image URI, inference instance type, and feature columns.

In [ ]:
model_metadata_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=SAGEMAKER_MODEL_METADATA_KEY,
)

model_metadata = json.load(model_metadata_obj["Body"])

In [ ]:
model_data_s3_uri = model_metadata["model_data_s3_uri"]
xgboost_image_uri = model_metadata["xgboost_image_uri"]
inference_instance_type = model_metadata["inference_instance_type"]
sagemaker_model_name = model_metadata["sagemaker_model_name"]

print(f"Model artifact: {model_data_s3_uri}")
print(f"Container image: {xgboost_image_uri}")
print(f"Inference instance type: {inference_instance_type}")
print(f"SageMaker Model: {sagemaker_model_name}")

## Load Latest Clarify Metadata

Load the latest Clarify metadata file from Amazon S3.

The Clarify output is referenced as bias and explainability metadata during model registration.

In [ ]:
clarify_metadata_response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=CLARIFY_METADATA_PREFIX,
)

clarify_metadata_files = clarify_metadata_response.get("Contents", [])

if not clarify_metadata_files:
    raise ValueError("No Clarify metadata files found.")

latest_clarify_metadata_key = sorted(
    clarify_metadata_files,
    key=lambda obj: obj["LastModified"],
)[-1]["Key"]

print(f"Latest Clarify metadata: s3://{BUCKET_NAME}/{latest_clarify_metadata_key}")

In [ ]:
clarify_metadata_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=latest_clarify_metadata_key,
)

clarify_metadata = json.load(clarify_metadata_obj["Body"])

## Locate Clarify Analysis Result

Find the generated Clarify `analysis.json` file.

This file is used as the bias and explainability report reference in the Model Registry.

In [ ]:
clarify_output_s3_uri = clarify_metadata["clarify_output_s3_uri"]
clarify_output_prefix = clarify_output_s3_uri.replace(
    f"s3://{BUCKET_NAME}/",
    "",
)

clarify_output_response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=clarify_output_prefix,
)

clarify_output_files = [
    obj["Key"]
    for obj in clarify_output_response.get("Contents", [])
]

analysis_json_keys = [
    key
    for key in clarify_output_files
    if key.endswith("analysis.json")
]

if not analysis_json_keys:
    raise ValueError("No Clarify analysis.json file found.")

clarify_analysis_key = analysis_json_keys[0]
clarify_analysis_s3_uri = f"s3://{BUCKET_NAME}/{clarify_analysis_key}"

print(f"Clarify analysis result: {clarify_analysis_s3_uri}")

## Create Model Package Group

Create the Model Package Group if it does not already exist.

The Model Package Group is the stable registry container for all versions of this model.

In [ ]:
try:
    model_package_group_description = sm.describe_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    )

    print("Model Package Group already exists.")
    print(model_package_group_description["ModelPackageGroupName"])

except ClientError as error:
    error_code = error.response["Error"]["Code"]

    if error_code == "ValidationException":
        create_group_response = sm.create_model_package_group(
            ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
            ModelPackageGroupDescription=MODEL_PACKAGE_GROUP_DESCRIPTION,
        )

        print("Created Model Package Group.")
        print(create_group_response["ModelPackageGroupArn"])
    else:
        raise

## Register Model Version

Register a new model version in the Model Package Group.

The registered version references the model artifact, inference container, evaluation report, and Clarify analysis result.

In [ ]:
evaluation_report_s3_uri = f"s3://{BUCKET_NAME}/{EVALUATION_REPORT_KEY}"

model_metrics = {
    "ModelQuality": {
        "Statistics": {
            "ContentType": "application/json",
            "S3Uri": evaluation_report_s3_uri,
        }
    },
    "Bias": {
        "Report": {
            "ContentType": "application/json",
            "S3Uri": clarify_analysis_s3_uri,
        }
    },
    "Explainability": {
        "Report": {
            "ContentType": "application/json",
            "S3Uri": clarify_analysis_s3_uri,
        }
    },
}

In [ ]:
create_model_package_response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelPackageDescription=(
        "German Credit Risk XGBoost model registered from the aws-sagemaker-showcase workflow."
    ),
    InferenceSpecification={
        "Containers": [
            {
                "Image": xgboost_image_uri,
                "ModelDataUrl": model_data_s3_uri,
            }
        ],
        "SupportedContentTypes": [
            "text/csv",
        ],
        "SupportedResponseMIMETypes": [
            "text/csv",
        ],
        "SupportedRealtimeInferenceInstanceTypes": [
            inference_instance_type,
        ],
        "SupportedTransformInstanceTypes": [
            inference_instance_type,
        ],
    },
    ModelMetrics=model_metrics,
    ModelApprovalStatus=MODEL_APPROVAL_STATUS,
    CustomerMetadataProperties={
        "project": "aws-sagemaker-showcase",
        "use_case": "german-credit-risk",
        "model_type": "xgboost",
        "training_mode": "local",
        "sagemaker_model_name": sagemaker_model_name,
        "clarify_job_name": clarify_metadata["clarify_job_name"],
    },
)

model_package_arn = create_model_package_response["ModelPackageArn"]

print(f"Registered model package: {model_package_arn}")

In [ ]:
model_package_description = sm.describe_model_package(
    ModelPackageName=model_package_arn,
)

print(f"Model package status: {model_package_description['ModelPackageStatus']}")
print(f"Approval status: {model_package_description['ModelApprovalStatus']}")

## Approve Registered Model

Approve the registered model package after reviewing the evaluation and Clarify results.

The approved model package can be used in the deployment notebook.

In [ ]:
sm.update_model_package(
    ModelPackageArn=model_package_arn,
    ModelApprovalStatus="Approved",
)

print(f"Approved model package: {model_package_arn}")

Approved model package: arn:aws:sagemaker:eu-central-1:591874026136:model-package/german-credit-risk-xgboost/1


In [ ]:
model_package_description = sm.describe_model_package(
    ModelPackageName=model_package_arn,
)

print(f"Approval status: {model_package_description['ModelApprovalStatus']}")

Approval status: Approved


## Store Registry Metadata

Store metadata about the registered model package locally and in Amazon S3.

The next notebook can use this metadata to deploy the registered model version.

In [ ]:
registry_metadata = {
    "model_package_group_name": MODEL_PACKAGE_GROUP_NAME,
    "model_package_arn": model_package_arn,
    "model_package_version": model_package_description["ModelPackageVersion"],
    "model_package_status": model_package_description["ModelPackageStatus"],
    "model_approval_status": model_package_description["ModelApprovalStatus"],
    "model_data_s3_uri": model_data_s3_uri,
    "xgboost_image_uri": xgboost_image_uri,
    "evaluation_report_s3_uri": evaluation_report_s3_uri,
    "clarify_analysis_s3_uri": clarify_analysis_s3_uri,
    "clarify_job_name": clarify_metadata["clarify_job_name"],
    "sagemaker_model_name": sagemaker_model_name,
    "created_at": timestamp,
}

with open(LOCAL_REGISTRY_METADATA_PATH, "w") as f:
    json.dump(registry_metadata, f, indent=2)

s3.upload_file(
    str(LOCAL_REGISTRY_METADATA_PATH),
    BUCKET_NAME,
    MODEL_REGISTRY_METADATA_KEY,
)

print(f"Uploaded registry metadata to s3://{BUCKET_NAME}/{MODEL_REGISTRY_METADATA_KEY}")

Uploaded registry metadata to s3://aws-sagemaker-showcase/models/registry/model_registry_metadata.json


## Result

The latest SageMaker-compatible XGBoost model was registered in the SageMaker Model Registry.

The registered model package references the current model artifact, evaluation report, and Clarify analysis results. The model package is currently set to `PendingManualApproval` and can be reviewed before deployment in the next workflow step.